Song Analysis — Zero-Shot NLI (10 custom labels) fork
=========================================================
Joins `00_titles.csv` (rank / region metadata) onto the Zero-Shot NLI (10 custom labels) emotion scores
from `04_classification.ipynb (§ 4A)` and drops only songs with no scoreable lyrics.

**This notebook is one of two parallel forks.** The zero-shot (10 custom labels,
`bart-large-mnli`) and GoEmotions (28 fixed labels, `roberta-base-go_emotions`)
classifiers use different models and label sets, so their outputs are NOT pooled
into a single table. Each fork carries its own 05.x → 06.x path:

| fork | classify | output | analyse | charts |
|---|---|---|---|---|
| zero-shot NLI | `04` § 4A | `04.1_*.csv` | `05.1` | `06.1` |
| GoEmotions    | `04` § 4B | `04.2_*.csv` | `05.2` | `06.2` |

Cross-classifier questions belong in `07_compare_classifiers.ipynb`, which is the
only place the two taxonomies are put side by side.

Shared scoring contract
-----------------------
Both classifiers live in `04_classification.ipynb` and share one contract
(see its header). This notebook depends on it:

- Scores are **independent per-label probabilities** in [0, 1] in both forks —
  no softmax across labels, so they do not sum to 1.
- `dominant_emotion == 'unclassified'` means **no scoreable lyrics**, not low
  confidence. That is the only thing dropped below, and it removes the *same*
  ~107 songs from both forks — so 05.1 and 05.2 carry the same song set.
- Confidence rides along as data: `dominant_score` (the winning score) and
  `low_confidence` (`dominant_score < MIN_CONFIDENCE`, 0.30 in both forks).
  Nothing here filters on it; 06.x reports it and you can filter downstream —
  identically across forks — if you want to.

Note that a shared contract makes the two *structurally* comparable; it does not
make raw magnitudes interchangeable. RoBERTa's sigmoids run systematically lower
than bart-mnli's entailment probabilities, so cross-fork reads should go through
the z-scored views in 06.x § 4.

In [0]:
pip install duckdb

In [0]:
import duckdb as dd
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

# ── Fork config ───────────────────────────────────────────────────────────────
CLASSIFIER   = "zeroshot"
LABEL        = "Zero-Shot NLI (10 custom labels)"
EMOTION_PATH = PROCESSED / "04.1_emotion_scores_zeroshot.csv"
OUTPUT_PATH  = PROCESSED / "05.1_titles_emotion_scores_zeroshot.csv"
TITLE_PATH   = PROCESSED / "00_titles.csv"


con = dd.connect()
con.execute(f"CREATE OR REPLACE TABLE top_titles AS SELECT * FROM read_csv_auto('{TITLE_PATH}')")
con.execute(f"CREATE OR REPLACE TABLE emotion_scores AS SELECT * FROM read_csv_auto('{EMOTION_PATH}')")

# Emotion columns are discovered from the file rather than hard-coded, so the
# same cell works for a 10-label and a 28-label taxonomy.
emotion_cols = [
    r[0] for r in con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_name = 'emotion_scores' AND column_name LIKE 'emotion_%' "
        "ORDER BY ordinal_position"
    ).fetchall()
]
print(f"{LABEL}: {len(emotion_cols)} emotion columns")
print(emotion_cols)

### Join region metadata and drop unclassified songs

In [0]:
score_select = ", ".join(f"b.{c}" for c in emotion_cols)

# The only rows dropped are 'unclassified' — songs with no scoreable lyrics.
# Low-confidence rows are KEPT and carried with their flag, so both forks hold
# the same song set and any confidence filter is applied later, identically.
df = con.execute(f"""
select
  a.rank,
  a.artist,
  a.title,
  a.region,
  a.spotify_uri,
  b.dominant_emotion,
  b.dominant_score,
  b.low_confidence,
  {score_select}
from top_titles a
left join emotion_scores b
  on a.spotify_uri = b.spotify_uri
where b.dominant_emotion is not null
  and b.dominant_emotion <> 'unclassified'
""").df()

n_titles = con.execute("select count(*) from top_titles").fetchone()[0]
n_lowconf = int(df["low_confidence"].sum())

print(f"Region-rows retained: {len(df)} / {n_titles}")
print(f"  dropped (no scoreable lyrics or unmatched): {n_titles - len(df)}")
print(f"  retained but flagged low_confidence (< 0.30): {n_lowconf} "
      f"({n_lowconf / len(df):.1%})")
print(f"  median dominant_score: {df['dominant_score'].median():.3f}")
display(df.head(20))

### Save

In [0]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")